# MAPP-2: ROVER + MLM(BERT) + морфология + CRF
Бірнеше ASR жүйесінің (kk-turbo VAD/chunk, ksc2, MMS) нәтижесін сөз бойынша туралап (confusion network), әр «слотта» дұрыс сөзді **CRF** таңдайды (дауыс беру орнына). Белгілер: дауыстар, MLM контекст бағасы, морфология, сөздік. Қауіпсіздік шегі τ тек train ішінде (ішкі CV) таңдалады → test-ке «қарамайды».

Алдымен `MAPP_ASR_KSC2_chunk_VAD.ipynb` және `MAPP_ASR_MMS_VAD.ipynb` іске қосылған болса — олар автоматты қосылады. Жоқ жүйелер өткізіліп кетеді.

In [ ]:
!pip -q install sklearn-crfsuite jiwer
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.1 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
%%writefile mapp2_core.py
import math, re, random
from collections import Counter
import jiwer, sklearn_crfsuite

EPS = "<eps>"; MAXC = 4

def norm_text(t):
    t = str(t).lower(); t = re.sub(r"[^\w\s]", "", t)
    return re.sub(r"\s+", " ", t).strip()

def _ngram_dedup(w, k):
    o = []; i = 0
    while i < len(w):
        if len(o) >= k and w[i:i + k] == o[-k:]: i += k; continue
        o.append(w[i]); i += 1
    return o

def dedup_full(t):
    w = norm_text(t).split(); o = []; run = 0
    for x in w:
        if o and o[-1] == x: continue
        o.append(x)
    for k in (3, 2): o = _ngram_dedup(o, k)
    return " ".join(o)

# ---------- ROVER ----------
def add_to_network(net, hyp, nsys):
    if not net: return [[w] for w in hyp]
    n, m = len(net), len(hyp); INF = 10**9
    D = [[INF]*(m+1) for _ in range(n+1)]; B = [[None]*(m+1) for _ in range(n+1)]; D[0][0] = 0
    for i in range(n+1):
        for j in range(m+1):
            d = D[i][j]
            if d >= INF: continue
            if i < n and j < m:
                c = d + (0 if hyp[j] in net[i] else 1)
                if c < D[i+1][j+1]: D[i+1][j+1], B[i+1][j+1] = c, "M"
            if i < n:
                c = d + (0 if all(w == EPS for w in net[i]) else 1)
                if c < D[i+1][j]: D[i+1][j], B[i+1][j] = c, "D"
            if j < m:
                c = d + 1
                if c < D[i][j+1]: D[i][j+1], B[i][j+1] = c, "I"
    out, i, j = [], n, m
    while i > 0 or j > 0:
        op = B[i][j]
        if op == "M": out.append(net[i-1] + [hyp[j-1]]); i -= 1; j -= 1
        elif op == "D": out.append(net[i-1] + [EPS]); i -= 1
        else: out.append([EPS]*nsys + [hyp[j-1]]); j -= 1
    return out[::-1]

def build_network(hyps):
    net = []
    for s, h in enumerate(hyps): net = add_to_network(net, h.split(), s)
    return net

def slot_candidates(slot, sw):
    score, first = Counter(), {}
    for s, w in enumerate(slot): score[w] += sw[s]; first.setdefault(w, s)
    cands = sorted(score, key=lambda w: (-score[w], first[w]))[:MAXC]
    tot = sum(sw)
    return [{"w": w, "votes": score[w]/tot, "from_sys0": int(slot[0] == w)} for w in cands]

def align_ref(cps, ref):
    n, m = len(cps), len(ref); INF = 10**9
    words = [[c["w"] for c in cs] for cs in cps]
    D = [[INF]*(m+1) for _ in range(n+1)]; B = [[None]*(m+1) for _ in range(n+1)]; D[0][0] = 0
    for i in range(n+1):
        for j in range(m+1):
            d = D[i][j]
            if d >= INF: continue
            if i < n and j < m:
                c = d + (0 if ref[j] in words[i] else 1)
                if c < D[i+1][j+1]: D[i+1][j+1], B[i+1][j+1] = c, "M"
            if i < n:
                c = d + (0 if EPS in words[i] else 1)
                if c < D[i+1][j]: D[i+1][j], B[i+1][j] = c, "D"
            if j < m:
                c = d + 1
                if c < D[i][j+1]: D[i][j+1], B[i][j+1] = c, "I"
    gold, i, j = [None]*n, n, m
    while i > 0 or j > 0:
        op = B[i][j]
        if op == "M": gold[i-1] = ref[j-1]; i -= 1; j -= 1
        elif op == "D": gold[i-1] = EPS; i -= 1
        else: j -= 1
    return [f"C{ws.index(g)}" if g in ws else "C0" for ws, g in zip(words, gold)]

# ---------- features ----------
class MLMScorer:
    def __init__(self, name="xlm-roberta-base", window=15):
        import torch
        from transformers import AutoTokenizer, AutoModelForMaskedLM
        self.torch = torch; self.tok = AutoTokenizer.from_pretrained(name)
        self.dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.m = AutoModelForMaskedLM.from_pretrained(name).to(self.dev).eval()
        self.window = window; self.cache = {}
    def score(self, left, cand, right):
        key = (tuple(left[-self.window:]), cand, tuple(right[:self.window]))
        if key in self.cache: return self.cache[key]
        t = self.tok
        L = t(" ".join(left[-self.window:]), add_special_tokens=False)["input_ids"] if left else []
        R = t(" ".join(right[:self.window]), add_special_tokens=False)["input_ids"] if right else []
        C = t(" " + cand, add_special_tokens=False)["input_ids"]
        ids = [t.cls_token_id] + L + [t.mask_token_id]*len(C) + R + [t.sep_token_id]
        with self.torch.inference_mode():
            lg = self.m(self.torch.tensor([ids], device=self.dev)).logits[0]
        lp = self.torch.log_softmax(lg[1+len(L):1+len(L)+len(C)], -1)
        s = float(sum(lp[k, c] for k, c in enumerate(C)) / max(1, len(C)))
        self.cache[key] = s; return s

PRON = {"мен":"1sg","сен":"2sg","біз":"1pl","сіз":"2pol","сендер":"2pl","сіздер":"2pol"}
SUFFIX = [("2pol",("дыңыз","діңіз","тыңыз","тіңіз","сыздар","сіздер","сыз","сіз")),
          ("1pl",("мыз","міз","пыз","піз","быз","біз","дық","дік","тық","тік","сақ","сек")),
          ("1sg",("мын","мін","пын","пін","бын","бін","дым","дім","тым","тім","сам","сем")),
          ("2sg",("сың","сің","дың","дің","тың","тің","саң","сең"))]
def person_of(w):
    if w in PRON: return None
    for p, s in SUFFIX:
        if w.endswith(s): return p
def default_morph(word, left, right):
    p = person_of(word); subj = next((PRON[w] for w in reversed(left[-6:]) if w in PRON), None)
    return {"agree": 1.0 if p == subj else -1.0} if p and subj else {}

def slot_features(cands, left, right, mlm, morph, lex):
    f = {"bias": 1.0, "n_cand": float(len(cands))}
    if len(cands) == 1: return f
    bs = {}
    if mlm:
        for k, c in enumerate(cands):
            if c["w"] != EPS: bs[k] = mlm.score(left, c["w"], right)
    top = max(bs.values()) if bs else 0.0
    for k, c in enumerate(cands):
        p = f"c{k}_"; f[p+"votes"] = c["votes"]; f[p+"sys0"] = float(c["from_sys0"])
        if c["w"] == EPS: f[p+"eps"] = 1.0; continue
        f[p+"len"] = min(len(c["w"]), 15)/15
        if k in bs:
            f[p+"mlm"] = bs[k]; f[p+"mlm_rel"] = bs[k]-top; f[p+"mlm_best"] = float(bs[k] == top)
        if lex is not None:
            f[p+"known"] = float(c["w"] in lex); f[p+"logfreq"] = math.log1p(lex.get(c["w"], 0))/15
        for k2, v in morph(c["w"], left, right).items(): f[p+"m_"+k2] = float(v)
    return f

def featurize(net, sw, mlm, morph, lex):
    cps = [slot_candidates(s, sw) for s in net]
    maj = [c[0]["w"] for c in cps]; feats = []
    for i, cands in enumerate(cps):
        left = [w for w in maj[max(0, i-20):i] if w != EPS]
        right = [w for w in maj[i+1:i+21] if w != EPS]
        feats.append(slot_features(cands, left, right, mlm, morph, lex))
    return cps, feats

def decode(cps, marg, tau):
    out = []
    for cands, m in zip(cps, marg):
        k = 0
        if len(cands) > 1:
            best = max((l for l in m if int(l[1:]) < len(cands)), key=m.get)
            if best != "C0" and m[best] >= tau: k = int(best[1:])
        if cands[k]["w"] != EPS: out.append(cands[k]["w"])
    return " ".join(out)

def vote_text(cps): return " ".join(c[0]["w"] for c in cps if c[0]["w"] != EPS)

def errs(ref, hyp):
    o = jiwer.process_words(ref, hyp if hyp.strip() else "<empty>")
    return o.substitutions + o.deletions + o.insertions

def train_crf(data, files):
    crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=200, all_possible_transitions=True)
    crf.fit([data[f][1] for f in files], [data[f][2] for f in files]); return crf

def split_folds(files, k, seed=0):
    o = files[:]; random.Random(seed).shuffle(o); return [o[i::k] for i in range(k)]

def choose_tau(data, refs, train, taus, inner=4, brk=2.0, min_gain=1):
    """τ по внутренней CV на train: максимизируем fix - brk*break (в слов-ошибках файла)."""
    folds = split_folds(train, inner, seed=1)
    margs = {}
    for te in folds:
        tr = [f for f in train if f not in set(te)]
        crf = train_crf(data, tr)
        for f in te: margs[f] = crf.predict_marginals_single(data[f][1])
    best, bt = 0.0, 1e9
    for t in taus:
        gain = 0.0
        for f in train:
            cps = data[f][0]; r = refs[f]
            base = errs(r, vote_text(cps)); new = errs(r, decode(cps, margs[f], t))
            d = base - new; gain += d if d > 0 else brk*d
        if gain > best + 1e-9 and gain >= min_gain: best, bt = gain, t
    return bt

def run_cv(files, refs, hyps, sw, mlm=None, morph=default_morph, lex=None,
           folds=5, taus=(0.4,0.5,0.6,0.7,0.8,0.9,0.95), log=print):
    data = {}
    for n, f in enumerate(files, 1):
        net = build_network(hyps[f]); cps, X = featurize(net, sw, mlm, morph, lex)
        data[f] = (cps, X, align_ref(cps, refs[f].split()))
        if n % 20 == 0: log(f"  features {n}/{len(files)}")
    out = {"raw": {}, "vote": {}, "crf_auto": {}, **{f"crf_{t}": {} for t in taus}}
    tau_used = []
    for k, te in enumerate(split_folds(files, folds)):
        tr = [f for f in files if f not in set(te)]
        crf = train_crf(data, tr); tau = choose_tau(data, refs, tr, taus); tau_used.append(tau)
        for f in te:
            cps, X, _ = data[f]; m = crf.predict_marginals_single(X)
            out["raw"][f] = hyps[f][0]; out["vote"][f] = vote_text(cps)
            out["crf_auto"][f] = decode(cps, m, tau)
            for t in taus: out[f"crf_{t}"][f] = decode(cps, m, t)
        log(f"  fold {k+1}/{folds} τ={tau}")
    out["oracle"] = {f: " ".join(c[int(l[1:])]["w"] for c, l in zip(data[f][0], data[f][2])
                                 if c[int(l[1:])]["w"] != EPS) for f in files}
    return out, tau_used, data

def summarize(files, refs, out):
    N = sum(len(refs[f].split()) for f in files)
    base = {f: errs(refs[f], out["raw"][f]) for f in files}
    rows = []
    for name, h in out.items():
        e = {f: errs(refs[f], h[f]) for f in files}
        cer = jiwer.cer([refs[f] for f in files], [h[f] or "x" for f in files])
        b = sum(e[f] < base[f] for f in files); w = sum(e[f] > base[f] for f in files)
        rows.append(dict(system=name, WER=sum(e.values())/N, CER=cer, better=b, worse=w, same=len(files)-b-w))
    return rows


# ---------- қазақша нормализация: цифр→сөз, бөгеу дыбыстар ----------
import re
ONES=["","бір","екі","үш","төрт","бес","алты","жеті","сегіз","тоғыз"]
TENS=["","он","жиырма","отыз","қырық","елу","алпыс","жетпіс","сексен","тоқсан"]
def _lt1000(n):
    w=[]
    if n>=100: w+= ([ONES[n//100]] if n//100>1 else [])+["жүз"]; n%=100
    if n>=10: w.append(TENS[n//10]); n%=10
    if n: w.append(ONES[n])
    return w
def num2kk(n):
    if n==0: return "нөл"
    parts=[]
    for div,name in ((10**9,"миллиард"),(10**6,"миллион"),(1000,"мың")):
        if n>=div:
            q=n//div; n%=div
            parts+= ([] if (name=="мың" and q==1) else _lt1000(q))+[name]
    parts+=_lt1000(n)
    return " ".join(parts)
FILLERS={"ыыы","ы","ыы","мхм","і","ііі","іі","ммм","мм","м","ым","ыым","хм","эмм","ээ"}
def digits_to_words(t):
    def rep(m):
        s=m.group(1)
        return num2kk(int(s)) if len(s)<=12 else s
    return re.sub(r"(\d+)[^\W\d_]*",lambda m: rep(m)+" ",t)   # '90шы' -> 'тоқсан' (қосымша тасталады)
def norm_eval(t, fillers=True, numerals=True):
    t=str(t).lower(); t=re.sub(r"[^\w\s]","",t)
    if numerals: t=digits_to_words(t)
    w=t.split()
    if fillers: w=[x for x in w if x not in FILLERS]
    return " ".join(w)


# ---------- қазақша нормализация: цифр→сөз, бөгеу дыбыстар ----------
import re
ONES=["","бір","екі","үш","төрт","бес","алты","жеті","сегіз","тоғыз"]
TENS=["","он","жиырма","отыз","қырық","елу","алпыс","жетпіс","сексен","тоқсан"]
def _lt1000(n):
    w=[]
    if n>=100: w+= ([ONES[n//100]] if n//100>1 else [])+["жүз"]; n%=100
    if n>=10: w.append(TENS[n//10]); n%=10
    if n: w.append(ONES[n])
    return w
def num2kk(n):
    if n==0: return "нөл"
    parts=[]
    for div,name in ((10**9,"миллиард"),(10**6,"миллион"),(1000,"мың")):
        if n>=div:
            q=n//div; n%=div
            parts+= ([] if (name=="мың" and q==1) else _lt1000(q))+[name]
    parts+=_lt1000(n)
    return " ".join(parts)
FILLERS={"ыыы","ы","ыы","мхм","і","ііі","іі","ммм","мм","м","ым","ыым","хм","эмм","ээ"}
def digits_to_words(t):
    def rep(m):
        s=m.group(1)
        return num2kk(int(s)) if len(s)<=12 else s
    return re.sub(r"(\d+)[^\W\d_]*",lambda m: rep(m)+" ",t)   # '90шы' -> 'тоқсан' (қосымша тасталады)
def norm_eval(t, fillers=True, numerals=True):
    t=str(t).lower(); t=re.sub(r"[^\w\s]","",t)
    if numerals: t=digits_to_words(t)
    w=t.split()
    if fillers: w=[x for x in w if x not in FILLERS]
    return " ".join(w)


# ---------- CSE морфология (түбір + жалғау талдаушы) ----------
import re as _re_morph
import numpy as np
from collections import defaultdict as _dd, Counter as _Counter

MOTION = {"бар","кел","кет","шық","кір","қайт","жет","ұш","жүр","жүгір","жүз","көш","аттан","қон","түс","өт","апар","әкел","әкет","орал","жылжы","көтеріл","жақында","ауыс"}
TRANSITIVE = {"көр","ал","бер","жаса","оқы","жаз","айт","же","іш","тап","сал","қой","аш","жап","тыңда","түсін","біл","ұста","сат","кес","жу","таңда","шеш","көтер","тексер","дайында","аяқта","баста","бітір","орында"}
_HOMO = {"i":"і","a":"а","e":"е","o":"о","p":"р","c":"с","x":"х","y":"у","k":"к","m":"м","t":"т","ј":"і","h":"һ"}
_VPAIR = {"а":"е","е":"а","о":"ө","ө":"о","ұ":"ү","ү":"ұ","ы":"і","і":"ы"}
def _mclean(s): return "".join(_HOMO.get(c, c) for c in str(s).strip().lower())
def _mtwin(e): return "".join(_VPAIR.get(c, c) for c in e)
CASES = ["nom","gen","dat","acc","loc","abl","ins"]
CASE_TAG = {"gen":1,"dat":2,"acc":3,"loc":4,"abl":5,"ınst":6,"ınstr":6,"instr":6,"inst":6}
FINITE = {"pst","per","tpat","ıft","tft","aft","apt","tpt","ppt","mdes","mimp","mcond"}

class Tagger:
    def __init__(self, stems, ana, votes):
        self.S, self.ana, self.votes = stems, ana, votes
        self.maxe = max((len(e) for e in ana), default=0); self._w = {}
    @classmethod
    def build(cls, stems_xlsx, endings_xlsx, uni_counter):
        import pandas as _pd
        st = _pd.read_excel(stems_xlsx, header=None)
        S = {_mclean(v) for v in st.iloc[:, 0].dropna()}
        S = {w for w in S if w and all(c.isalpha() or c == "-" for c in w)}
        en = _pd.read_excel(endings_xlsx, header=None, skiprows=1)
        ana = _dd(set)
        for e, d in zip(en.iloc[:, 0], en.iloc[:, 1]):
            if not isinstance(e, str) or not isinstance(d, str): continue
            tags = frozenset(_re_morph.findall(r"<([^>]+)>", d))
            if not tags: continue
            key = _mclean(e)
            for k in (key, _mtwin(key)):
                if k: ana[k].add(tags)
        ana = {k: list(v) for k, v in ana.items()}
        votes = cls._harvest(S, ana, uni_counter)
        print("CSE: түбір %d, жалғау-үлгі %d" % (len(S), len(ana)))
        return cls(S, ana, votes)
    @staticmethod
    def _harvest(S, ana, uni_counter):
        vb = {e for e, a in ana.items() if all("VB" in t for t in a)}
        nb = {e for e, a in ana.items() if all("NB" in t for t in a)}
        maxe = max((len(e) for e in ana), default=0)
        votes = _dd(lambda: [0, 0])
        for w, n in uni_counter.items():
            if len(w) < 4 or not w.isalpha(): continue
            for k in range(len(w) - 1, 1, -1):
                if len(w) - k > maxe: break
                stem, end = w[:k], w[k:]
                if stem not in S: continue
                if end in vb: votes[stem][0] += n
                elif end in nb: votes[stem][1] += n
        return {k: tuple(v) for k, v in votes.items()}
    def stem_pos(self, s):
        v, n = self.votes.get(s, (0, 0)); t = v + n
        if t < 5: return 2
        r = v / t
        return 1 if r > 0.6 else (0 if r < 0.4 else 2)
    def word(self, w):
        t = self._w.get(w)
        if t is None: t = self._analyze(w); self._w[w] = t
        return t
    def _analyze(self, w):
        if not w or not w.isalpha(): return (2, 0, False, False, False, False, False)
        if w in self.S:
            p = self.stem_pos(w); return (p, 0, False, p == 1, w in MOTION, w in TRANSITIVE, True)
        best, best_sc = None, -1e9
        for k in range(len(w) - 1, 1, -1):
            if len(w) - k > self.maxe: break
            stem, end = w[:k], w[k:]
            if stem not in self.S: continue
            for tags in self.ana.get(end, ()):
                sp = self.stem_pos(stem); sc = 0.1 * len(end)
                if "VB" in tags: sc += 2.0 if sp == 1 else (-1.0 if sp == 0 else 0.0)
                if "NB" in tags: sc += 2.0 if sp == 0 else (-1.0 if sp == 1 else 0.0)
                if sc > best_sc: best, best_sc = (stem, tags, sp), sc
        if best is None: return (2, 0, False, False, False, False, False)
        stem, tags, sp = best
        pos = 1 if ("VB" in tags and "NB" not in tags) else (0 if ("NB" in tags and "VB" not in tags) else sp)
        case = 0
        for t in tags:
            if t in CASE_TAG: case = CASE_TAG[t]; break
        poss = "pos" in tags
        fin = pos == 1 and bool(tags & FINITE) and "pp" not in tags
        return (pos, case, poss, fin, stem in MOTION, stem in TRANSITIVE, True)

CSE_TR_NAMES = (["pos_%s_%s" % (a, b) for a in range(3) for b in range(3)]
                + ["%s_poss" % c for c in CASES] + ["%s_verb" % c for c in CASES]
                + ["dat_motion", "acc_trans", "nom_fin", "same_case", "both_bad", "gen_nonposs"])

def _pair_feats(a, b):
    """(алдыңғы сөз, келесі сөз) талдауынан GEN+POSS, DAT+MOTION, ACC+TRANS т.б. белгілер."""
    pa, ca, _, _, _, _, oka = a; pb, cb, possb, finb, motb, trb, okb = b
    f = {}
    f["pos_%d_%d" % (pa, pb)] = 1.0
    if possb: f["%s_poss" % CASES[ca]] = 1.0
    if pb == 1: f["%s_verb" % CASES[ca]] = 1.0
    if ca == 2 and motb: f["dat_motion"] = 1.0
    if ca == 3 and trb: f["acc_trans"] = 1.0
    if pa == 0 and ca == 0 and finb: f["nom_fin"] = 1.0
    if ca == cb and ca != 0: f["same_case"] = 1.0
    if not oka and not okb: f["both_bad"] = 1.0
    if ca == 1 and not possb: f["gen_nonposs"] = 1.0
    return f

def make_cse_morph(tagger, base_morph=None):
    """MAPP-2 morph hook: morph(word, left, right) -> dict. Кандидат сөздің өз талдауы + сол/оң көрші сөзбен CSE жұп-белгілері."""
    def morph(word, left, right):
        out = dict(base_morph(word, left, right)) if base_morph else {}
        t = tagger.word(word); pos, case, poss, fin, mot, tr, ok = t
        out["cse_ok"] = float(ok); out["cse_pos%d" % pos] = 1.0
        if case: out["cse_case%d" % case] = 1.0
        if poss: out["cse_poss"] = 1.0
        if fin: out["cse_fin"] = 1.0
        if left:
            for k, v in _pair_feats(tagger.word(left[-1]), t).items(): out["L_" + k] = v
        if right:
            for k, v in _pair_feats(t, tagger.word(right[0])).items(): out["R_" + k] = v
        return out
    return morph


# ---------- қазақша нормализация: цифр→сөз, бөгеу дыбыстар ----------
import re
ONES=["","бір","екі","үш","төрт","бес","алты","жеті","сегіз","тоғыз"]
TENS=["","он","жиырма","отыз","қырық","елу","алпыс","жетпіс","сексен","тоқсан"]
def _lt1000(n):
    w=[]
    if n>=100: w+= ([ONES[n//100]] if n//100>1 else [])+["жүз"]; n%=100
    if n>=10: w.append(TENS[n//10]); n%=10
    if n: w.append(ONES[n])
    return w
def num2kk(n):
    if n==0: return "нөл"
    parts=[]
    for div,name in ((10**9,"миллиард"),(10**6,"миллион"),(1000,"мың")):
        if n>=div:
            q=n//div; n%=div
            parts+= ([] if (name=="мың" and q==1) else _lt1000(q))+[name]
    parts+=_lt1000(n)
    return " ".join(parts)
FILLERS={"ыыы","ы","ыы","мхм","і","ііі","іі","ммм","мм","м","ым","ыым","хм","эмм","ээ"}
def digits_to_words(t):
    def rep(m):
        s=m.group(1)
        return num2kk(int(s)) if len(s)<=12 else s
    return re.sub(r"(\d+)[^\W\d_]*",lambda m: rep(m)+" ",t)   # '90шы' -> 'тоқсан' (қосымша тасталады)
def norm_eval(t, fillers=True, numerals=True):
    t=str(t).lower(); t=re.sub(r"[^\w\s]","",t)
    if numerals: t=digits_to_words(t)
    w=t.split()
    if fillers: w=[x for x in w if x not in FILLERS]
    return " ".join(w)


# ---------- CSE морфология (түбір + жалғау талдаушы) ----------
import re as _re_morph
import numpy as np
from collections import defaultdict as _dd, Counter as _Counter

MOTION = {"бар","кел","кет","шық","кір","қайт","жет","ұш","жүр","жүгір","жүз","көш","аттан","қон","түс","өт","апар","әкел","әкет","орал","жылжы","көтеріл","жақында","ауыс"}
TRANSITIVE = {"көр","ал","бер","жаса","оқы","жаз","айт","же","іш","тап","сал","қой","аш","жап","тыңда","түсін","біл","ұста","сат","кес","жу","таңда","шеш","көтер","тексер","дайында","аяқта","баста","бітір","орында"}
_HOMO = {"i":"і","a":"а","e":"е","o":"о","p":"р","c":"с","x":"х","y":"у","k":"к","m":"м","t":"т","ј":"і","h":"һ"}
_VPAIR = {"а":"е","е":"а","о":"ө","ө":"о","ұ":"ү","ү":"ұ","ы":"і","і":"ы"}
def _mclean(s): return "".join(_HOMO.get(c, c) for c in str(s).strip().lower())
def _mtwin(e): return "".join(_VPAIR.get(c, c) for c in e)
CASES = ["nom","gen","dat","acc","loc","abl","ins"]
CASE_TAG = {"gen":1,"dat":2,"acc":3,"loc":4,"abl":5,"ınst":6,"ınstr":6,"instr":6,"inst":6}
FINITE = {"pst","per","tpat","ıft","tft","aft","apt","tpt","ppt","mdes","mimp","mcond"}

class Tagger:
    def __init__(self, stems, ana, votes):
        self.S, self.ana, self.votes = stems, ana, votes
        self.maxe = max((len(e) for e in ana), default=0); self._w = {}
    @classmethod
    def build(cls, stems_xlsx, endings_xlsx, uni_counter):
        import pandas as _pd
        st = _pd.read_excel(stems_xlsx, header=None)
        S = {_mclean(v) for v in st.iloc[:, 0].dropna()}
        S = {w for w in S if w and all(c.isalpha() or c == "-" for c in w)}
        en = _pd.read_excel(endings_xlsx, header=None, skiprows=1)
        ana = _dd(set)
        for e, d in zip(en.iloc[:, 0], en.iloc[:, 1]):
            if not isinstance(e, str) or not isinstance(d, str): continue
            tags = frozenset(_re_morph.findall(r"<([^>]+)>", d))
            if not tags: continue
            key = _mclean(e)
            for k in (key, _mtwin(key)):
                if k: ana[k].add(tags)
        ana = {k: list(v) for k, v in ana.items()}
        votes = cls._harvest(S, ana, uni_counter)
        print("CSE: түбір %d, жалғау-үлгі %d" % (len(S), len(ana)))
        return cls(S, ana, votes)
    @staticmethod
    def _harvest(S, ana, uni_counter):
        vb = {e for e, a in ana.items() if all("VB" in t for t in a)}
        nb = {e for e, a in ana.items() if all("NB" in t for t in a)}
        maxe = max((len(e) for e in ana), default=0)
        votes = _dd(lambda: [0, 0])
        for w, n in uni_counter.items():
            if len(w) < 4 or not w.isalpha(): continue
            for k in range(len(w) - 1, 1, -1):
                if len(w) - k > maxe: break
                stem, end = w[:k], w[k:]
                if stem not in S: continue
                if end in vb: votes[stem][0] += n
                elif end in nb: votes[stem][1] += n
        return {k: tuple(v) for k, v in votes.items()}
    def stem_pos(self, s):
        v, n = self.votes.get(s, (0, 0)); t = v + n
        if t < 5: return 2
        r = v / t
        return 1 if r > 0.6 else (0 if r < 0.4 else 2)
    def word(self, w):
        t = self._w.get(w)
        if t is None: t = self._analyze(w); self._w[w] = t
        return t
    def _analyze(self, w):
        if not w or not w.isalpha(): return (2, 0, False, False, False, False, False)
        if w in self.S:
            p = self.stem_pos(w); return (p, 0, False, p == 1, w in MOTION, w in TRANSITIVE, True)
        best, best_sc = None, -1e9
        for k in range(len(w) - 1, 1, -1):
            if len(w) - k > self.maxe: break
            stem, end = w[:k], w[k:]
            if stem not in self.S: continue
            for tags in self.ana.get(end, ()):
                sp = self.stem_pos(stem); sc = 0.1 * len(end)
                if "VB" in tags: sc += 2.0 if sp == 1 else (-1.0 if sp == 0 else 0.0)
                if "NB" in tags: sc += 2.0 if sp == 0 else (-1.0 if sp == 1 else 0.0)
                if sc > best_sc: best, best_sc = (stem, tags, sp), sc
        if best is None: return (2, 0, False, False, False, False, False)
        stem, tags, sp = best
        pos = 1 if ("VB" in tags and "NB" not in tags) else (0 if ("NB" in tags and "VB" not in tags) else sp)
        case = 0
        for t in tags:
            if t in CASE_TAG: case = CASE_TAG[t]; break
        poss = "pos" in tags
        fin = pos == 1 and bool(tags & FINITE) and "pp" not in tags
        return (pos, case, poss, fin, stem in MOTION, stem in TRANSITIVE, True)

CSE_TR_NAMES = (["pos_%s_%s" % (a, b) for a in range(3) for b in range(3)]
                + ["%s_poss" % c for c in CASES] + ["%s_verb" % c for c in CASES]
                + ["dat_motion", "acc_trans", "nom_fin", "same_case", "both_bad", "gen_nonposs"])

def _pair_feats(a, b):
    """(алдыңғы сөз, келесі сөз) талдауынан GEN+POSS, DAT+MOTION, ACC+TRANS т.б. белгілер."""
    pa, ca, _, _, _, _, oka = a; pb, cb, possb, finb, motb, trb, okb = b
    f = {}
    f["pos_%d_%d" % (pa, pb)] = 1.0
    if possb: f["%s_poss" % CASES[ca]] = 1.0
    if pb == 1: f["%s_verb" % CASES[ca]] = 1.0
    if ca == 2 and motb: f["dat_motion"] = 1.0
    if ca == 3 and trb: f["acc_trans"] = 1.0
    if pa == 0 and ca == 0 and finb: f["nom_fin"] = 1.0
    if ca == cb and ca != 0: f["same_case"] = 1.0
    if not oka and not okb: f["both_bad"] = 1.0
    if ca == 1 and not possb: f["gen_nonposs"] = 1.0
    return f

def make_cse_morph(tagger, base_morph=None):
    """MAPP-2 morph hook: morph(word, left, right) -> dict. Кандидат сөздің өз талдауы + сол/оң көрші сөзбен CSE жұп-белгілері."""
    def morph(word, left, right):
        out = dict(base_morph(word, left, right)) if base_morph else {}
        t = tagger.word(word); pos, case, poss, fin, mot, tr, ok = t
        out["cse_ok"] = float(ok); out["cse_pos%d" % pos] = 1.0
        if case: out["cse_case%d" % case] = 1.0
        if poss: out["cse_poss"] = 1.0
        if fin: out["cse_fin"] = 1.0
        if left:
            for k, v in _pair_feats(tagger.word(left[-1]), t).items(): out["L_" + k] = v
        if right:
            for k, v in _pair_feats(t, tagger.word(right[0])).items(): out["R_" + k] = v
        return out
    return morph


# ---------- биграмма ЯМ (корпустан) ----------
import math
from collections import Counter

class BigramLM:
    """Корпустан биграмма ЯМ. MLMScorer-мен бірдей интерфейс: score(left, cand, right) -> log P(cand|left) + log P(right|cand).
    Тек гипотезалардағы сөздерге қатысты биграммалар сақталады (жад үнемі)."""
    def __init__(self, corpus_path, vocab, norm, lam=0.7, max_lines=None, log=print):
        self.uni, self.bi = Counter(), Counter(); V = set(vocab); self.lam = lam
        n = 0
        with open(corpus_path, encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                w = norm(line).split()
                if not w: continue
                self.uni.update(w)
                prev = "<s>"
                for x in w:
                    if prev in V or x in V: self.bi[(prev, x)] += 1
                    prev = x
                if prev in V: self.bi[(prev, "</s>")] += 1
                n += 1
                if max_lines and n >= max_lines: break
        self.ctx = Counter()
        for (a, b), c in self.bi.items(): self.ctx[a] += c
        self.N = sum(self.uni.values()); self.Vn = len(self.uni) + 1
        self.uni["<s>"] = n
        log(f"ЯМ: {n} сөйлем, {self.N} сөз, сөздік {len(self.uni)}, биграмма {len(self.bi)}")
    def p(self, prev, w):
        pu = (self.uni.get(w, 0) + 0.5) / (self.N + 0.5 * self.Vn)
        cp = self.uni.get(prev, 0) if prev != "<s>" else self.uni["<s>"]
        if prev in self.ctx or prev == "<s>":
            tot = self.ctx.get(prev, 0)
            pb = self.bi.get((prev, w), 0) / tot if tot else 0.0
            return math.log(self.lam * pb + (1 - self.lam) * pu)
        return math.log(pu)
    def score(self, left, cand, right):
        l = left[-1] if left else "<s>"
        s = self.p(l, cand)
        s += self.p(cand, right[0]) if right else self.p(cand, "</s>")
        return s


# ---------- қазақша нормализация: цифр→сөз, бөгеу дыбыстар ----------
import re
ONES=["","бір","екі","үш","төрт","бес","алты","жеті","сегіз","тоғыз"]
TENS=["","он","жиырма","отыз","қырық","елу","алпыс","жетпіс","сексен","тоқсан"]
def _lt1000(n):
    w=[]
    if n>=100: w+= ([ONES[n//100]] if n//100>1 else [])+["жүз"]; n%=100
    if n>=10: w.append(TENS[n//10]); n%=10
    if n: w.append(ONES[n])
    return w
def num2kk(n):
    if n==0: return "нөл"
    parts=[]
    for div,name in ((10**9,"миллиард"),(10**6,"миллион"),(1000,"мың")):
        if n>=div:
            q=n//div; n%=div
            parts+= ([] if (name=="мың" and q==1) else _lt1000(q))+[name]
    parts+=_lt1000(n)
    return " ".join(parts)
FILLERS={"ыыы","ы","ыы","мхм","і","ііі","іі","ммм","мм","м","ым","ыым","хм","эмм","ээ"}
def digits_to_words(t):
    def rep(m):
        s=m.group(1)
        return num2kk(int(s)) if len(s)<=12 else s
    return re.sub(r"(\d+)[^\W\d_]*",lambda m: rep(m)+" ",t)   # '90шы' -> 'тоқсан' (қосымша тасталады)
def norm_eval(t, fillers=True, numerals=True):
    t=str(t).lower(); t=re.sub(r"[^\w\s]","",t)
    if numerals: t=digits_to_words(t)
    w=t.split()
    if fillers: w=[x for x in w if x not in FILLERS]
    return " ".join(w)


# ---------- CSE морфология (түбір + жалғау талдаушы) ----------
import re as _re_morph
import numpy as np
from collections import defaultdict as _dd, Counter as _Counter

MOTION = {"бар","кел","кет","шық","кір","қайт","жет","ұш","жүр","жүгір","жүз","көш","аттан","қон","түс","өт","апар","әкел","әкет","орал","жылжы","көтеріл","жақында","ауыс"}
TRANSITIVE = {"көр","ал","бер","жаса","оқы","жаз","айт","же","іш","тап","сал","қой","аш","жап","тыңда","түсін","біл","ұста","сат","кес","жу","таңда","шеш","көтер","тексер","дайында","аяқта","баста","бітір","орында"}
_HOMO = {"i":"і","a":"а","e":"е","o":"о","p":"р","c":"с","x":"х","y":"у","k":"к","m":"м","t":"т","ј":"і","h":"һ"}
_VPAIR = {"а":"е","е":"а","о":"ө","ө":"о","ұ":"ү","ү":"ұ","ы":"і","і":"ы"}
def _mclean(s): return "".join(_HOMO.get(c, c) for c in str(s).strip().lower())
def _mtwin(e): return "".join(_VPAIR.get(c, c) for c in e)
CASES = ["nom","gen","dat","acc","loc","abl","ins"]
CASE_TAG = {"gen":1,"dat":2,"acc":3,"loc":4,"abl":5,"ınst":6,"ınstr":6,"instr":6,"inst":6}
FINITE = {"pst","per","tpat","ıft","tft","aft","apt","tpt","ppt","mdes","mimp","mcond"}

class Tagger:
    def __init__(self, stems, ana, votes):
        self.S, self.ana, self.votes = stems, ana, votes
        self.maxe = max((len(e) for e in ana), default=0); self._w = {}
    @classmethod
    def build(cls, stems_xlsx, endings_xlsx, uni_counter):
        import pandas as _pd
        st = _pd.read_excel(stems_xlsx, header=None)
        S = {_mclean(v) for v in st.iloc[:, 0].dropna()}
        S = {w for w in S if w and all(c.isalpha() or c == "-" for c in w)}
        en = _pd.read_excel(endings_xlsx, header=None, skiprows=1)
        ana = _dd(set)
        for e, d in zip(en.iloc[:, 0], en.iloc[:, 1]):
            if not isinstance(e, str) or not isinstance(d, str): continue
            tags = frozenset(_re_morph.findall(r"<([^>]+)>", d))
            if not tags: continue
            key = _mclean(e)
            for k in (key, _mtwin(key)):
                if k: ana[k].add(tags)
        ana = {k: list(v) for k, v in ana.items()}
        votes = cls._harvest(S, ana, uni_counter)
        print("CSE: түбір %d, жалғау-үлгі %d" % (len(S), len(ana)))
        return cls(S, ana, votes)
    @staticmethod
    def _harvest(S, ana, uni_counter):
        vb = {e for e, a in ana.items() if all("VB" in t for t in a)}
        nb = {e for e, a in ana.items() if all("NB" in t for t in a)}
        maxe = max((len(e) for e in ana), default=0)
        votes = _dd(lambda: [0, 0])
        for w, n in uni_counter.items():
            if len(w) < 4 or not w.isalpha(): continue
            for k in range(len(w) - 1, 1, -1):
                if len(w) - k > maxe: break
                stem, end = w[:k], w[k:]
                if stem not in S: continue
                if end in vb: votes[stem][0] += n
                elif end in nb: votes[stem][1] += n
        return {k: tuple(v) for k, v in votes.items()}
    def stem_pos(self, s):
        v, n = self.votes.get(s, (0, 0)); t = v + n
        if t < 5: return 2
        r = v / t
        return 1 if r > 0.6 else (0 if r < 0.4 else 2)
    def word(self, w):
        t = self._w.get(w)
        if t is None: t = self._analyze(w); self._w[w] = t
        return t
    def _analyze(self, w):
        if not w or not w.isalpha(): return (2, 0, False, False, False, False, False)
        if w in self.S:
            p = self.stem_pos(w); return (p, 0, False, p == 1, w in MOTION, w in TRANSITIVE, True)
        best, best_sc = None, -1e9
        for k in range(len(w) - 1, 1, -1):
            if len(w) - k > self.maxe: break
            stem, end = w[:k], w[k:]
            if stem not in self.S: continue
            for tags in self.ana.get(end, ()):
                sp = self.stem_pos(stem); sc = 0.1 * len(end)
                if "VB" in tags: sc += 2.0 if sp == 1 else (-1.0 if sp == 0 else 0.0)
                if "NB" in tags: sc += 2.0 if sp == 0 else (-1.0 if sp == 1 else 0.0)
                if sc > best_sc: best, best_sc = (stem, tags, sp), sc
        if best is None: return (2, 0, False, False, False, False, False)
        stem, tags, sp = best
        pos = 1 if ("VB" in tags and "NB" not in tags) else (0 if ("NB" in tags and "VB" not in tags) else sp)
        case = 0
        for t in tags:
            if t in CASE_TAG: case = CASE_TAG[t]; break
        poss = "pos" in tags
        fin = pos == 1 and bool(tags & FINITE) and "pp" not in tags
        return (pos, case, poss, fin, stem in MOTION, stem in TRANSITIVE, True)

CSE_TR_NAMES = (["pos_%s_%s" % (a, b) for a in range(3) for b in range(3)]
                + ["%s_poss" % c for c in CASES] + ["%s_verb" % c for c in CASES]
                + ["dat_motion", "acc_trans", "nom_fin", "same_case", "both_bad", "gen_nonposs"])

def _pair_feats(a, b):
    """(алдыңғы сөз, келесі сөз) талдауынан GEN+POSS, DAT+MOTION, ACC+TRANS т.б. белгілер."""
    pa, ca, _, _, _, _, oka = a; pb, cb, possb, finb, motb, trb, okb = b
    f = {}
    f["pos_%d_%d" % (pa, pb)] = 1.0
    if possb: f["%s_poss" % CASES[ca]] = 1.0
    if pb == 1: f["%s_verb" % CASES[ca]] = 1.0
    if ca == 2 and motb: f["dat_motion"] = 1.0
    if ca == 3 and trb: f["acc_trans"] = 1.0
    if pa == 0 and ca == 0 and finb: f["nom_fin"] = 1.0
    if ca == cb and ca != 0: f["same_case"] = 1.0
    if not oka and not okb: f["both_bad"] = 1.0
    if ca == 1 and not possb: f["gen_nonposs"] = 1.0
    return f

def make_cse_morph(tagger, base_morph=None):
    """MAPP-2 morph hook: morph(word, left, right) -> dict. Кандидат сөздің өз талдауы + сол/оң көрші сөзбен CSE жұп-белгілері."""
    def morph(word, left, right):
        out = dict(base_morph(word, left, right)) if base_morph else {}
        t = tagger.word(word); pos, case, poss, fin, mot, tr, ok = t
        out["cse_ok"] = float(ok); out["cse_pos%d" % pos] = 1.0
        if case: out["cse_case%d" % case] = 1.0
        if poss: out["cse_poss"] = 1.0
        if fin: out["cse_fin"] = 1.0
        if left:
            for k, v in _pair_feats(tagger.word(left[-1]), t).items(): out["L_" + k] = v
        if right:
            for k, v in _pair_feats(t, tagger.word(right[0])).items(): out["R_" + k] = v
        return out
    return morph


# ---------- биграмма ЯМ (корпустан) ----------
import math
from collections import Counter

class BigramLM:
    """Корпустан биграмма ЯМ. MLMScorer-мен бірдей интерфейс: score(left, cand, right) -> log P(cand|left) + log P(right|cand).
    Тек гипотезалардағы сөздерге қатысты биграммалар сақталады (жад үнемі)."""
    def __init__(self, corpus_path, vocab, norm, lam=0.7, max_lines=None, log=print):
        self.uni, self.bi = Counter(), Counter(); V = set(vocab); self.lam = lam
        n = 0
        with open(corpus_path, encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                w = norm(line).split()
                if not w: continue
                self.uni.update(w)
                prev = "<s>"
                for x in w:
                    if prev in V or x in V: self.bi[(prev, x)] += 1
                    prev = x
                if prev in V: self.bi[(prev, "</s>")] += 1
                n += 1
                if max_lines and n >= max_lines: break
        self.ctx = Counter()
        for (a, b), c in self.bi.items(): self.ctx[a] += c
        self.N = sum(self.uni.values()); self.Vn = len(self.uni) + 1
        self.uni["<s>"] = n
        log(f"ЯМ: {n} сөйлем, {self.N} сөз, сөздік {len(self.uni)}, биграмма {len(self.bi)}")
    def p(self, prev, w):
        pu = (self.uni.get(w, 0) + 0.5) / (self.N + 0.5 * self.Vn)
        cp = self.uni.get(prev, 0) if prev != "<s>" else self.uni["<s>"]
        if prev in self.ctx or prev == "<s>":
            tot = self.ctx.get(prev, 0)
            pb = self.bi.get((prev, w), 0) / tot if tot else 0.0
            return math.log(self.lam * pb + (1 - self.lam) * pu)
        return math.log(pu)
    def score(self, left, cand, right):
        l = left[-1] if left else "<s>"
        s = self.p(l, cand)
        s += self.p(cand, right[0]) if right else self.p(cand, "</s>")
        return s


Writing mapp2_core.py


In [ ]:
import os, glob, pandas as pd, numpy as np, time, sys
sys.path.insert(0, "/content")
from mapp2_core import *
_cands = ["/content/drive/MyDrive/Colab Notebooks/dataset_wav16k", "/content/drive/MyDrive/dataset_wav16k"] + glob.glob("/content/drive/MyDrive/**/dataset_wav16k", recursive=True)
BASE = next((c for c in _cands if os.path.isdir(c) and (glob.glob(c + "/*.csv") or os.path.isdir(c + "/ksc2"))), _cands[0])
print("BASE:", BASE)
OUT_DIR = os.path.join(BASE, "mapp2"); os.makedirs(OUT_DIR, exist_ok=True)

# (атау, файл, салмақ). ПЕРВЫЙ = негізгі жүйе (pivot, ең күшті RAW).
SYSTEMS = [
    ("ksc2_vad",      "ksc2/mapp_v2_batch_results_ksc2_vad.csv",    1.0),
    ("kkturbo_vad",   "mapp_v2_batch_results_kkturbo_vad.csv",      0.9),
    ("kkturbo_chunk", "mapp_v2_batch_results_kkturbo_chunk.csv",    0.8),
    ("ksc2_chunk",    "ksc2/mapp_v2_batch_results_ksc2_chunk.csv",  0.7),
    ("mms_vad",       "mms/mapp_v2_batch_results_mms_vad.csv",      0.7),
    ("ctc2_vad",      "ctc2/mapp_v2_batch_results_xlsr_kk_vad.csv", 0.6),   # қосымша CTC жүйесі (жоқ болса өткізіледі)
]
USE_MLM   = False                # MLM әзірге өшірулі (Stage 6 жасалмайды); True = MLM белгісін қосу
MLM_MODEL = "xlm-roberta-base"
FOLDS = 5

# CSV жоқ болса, xlsx-тің per_file парағынан оқиды (glob арқылы Drive ішінен іздейді)
ALT = {
    "ksc2_vad":      ("ksc2/mapp_asr_ksc2_comparison.xlsx", "ksc2_vad_per_file"),
    "ksc2_chunk":    ("ksc2/mapp_asr_ksc2_comparison.xlsx", "ksc2_chunk_per_file"),
    "mms_vad":       ("mms/mapp_asr_mms_comparison.xlsx",   "mms_per_file"),
    "kkturbo_chunk": ("mapp_asr_kkturbo_vad_comparison.xlsx", "kkturbo_chunk_per_file"),
}
def load(name, path):
    p = os.path.join(BASE, path)
    if os.path.exists(p):
        d = pd.read_csv(p)
    elif name in ALT:
        hits = glob.glob(os.path.join(BASE, "**", os.path.basename(ALT[name][0])), recursive=True)
        if not hits: print("  жоқ, өткізілді:", path); return None
        d = pd.read_excel(hits[0], sheet_name=ALT[name][1]); print("  xlsx-тен оқылды:", hits[0])
    else:
        print("  жоқ, өткізілді:", path); return None
    return {(str(r.duration), str(r.stem)): r for r in d.itertuples()}

SRC = {}
USE_CTC2 = False   # xlsr-kk пайда бермеді (0.1795 vs 0.1783) → негізгі нұсқада өшірулі
for name, path, w in SYSTEMS:
    if name == "ctc2_vad" and not USE_CTC2: continue
    s = load(name, path)
    if s is not None: SRC[name] = (s, w); print("OK", name, len(s))
names = list(SRC)
assert len(names) >= 2, "кемінде 2 жүйе керек"
keys = [k for k in SRC[names[0]][0] if all(k in SRC[n][0] for n in names)]
files = [f"{a}/{b}" for a, b in keys]
pivot = SRC[names[0]][0]
NORM_FILLERS = True    # «ыыы, мхм, ііі…» бөгеу дыбыстарды ref және гипотезадан алып тастау
NORM_NUM     = True    # цифр → қазақша сөз (екі жақта бірдей)
EXCLUDE      = []   # ref түзетілді, шығарылатын файл жоқ. (қажет болса: ["10m/10m_009"])
def get_ref(a, b):
    """Жаңартылған анықтама: references/<ұзындық>/<файл>.txt (CSV ішіндегі ref ескірген болуы мүмкін)."""
    p = os.path.join(BASE, "references", a, b + ".txt")
    if os.path.exists(p): return open(p, encoding="utf-8").read().strip(), True
    return str(pivot[(a, b)].ref), False
_r = {k: get_ref(*k) for k in keys}
refs = {f"{a}/{b}": norm_eval(_r[(a, b)][0], NORM_FILLERS, NORM_NUM) for a, b in keys}
_chg = [f"{a}/{b}" for a, b in keys if norm_eval(_r[(a, b)][0], NORM_FILLERS, NORM_NUM) != norm_eval(pivot[(a, b)].ref, NORM_FILLERS, NORM_NUM)]
print("references/*.txt-тен оқылды:", sum(v[1] for v in _r.values()), "/", len(keys), "| CSV-дегі ref-тен өзгерген:", _chg)
print("ортақ файл:", len(files), "| жүйелер:", names)

BASE: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k
OK ksc2_vad 100
OK kkturbo_vad 100
OK kkturbo_chunk 100
OK ksc2_chunk 100
OK mms_vad 100
references/*.txt-тен оқылды: 100 / 100 | CSV-дегі ref-тен өзгерген: ['10m/10m_009']
ортақ файл: 100 | жүйелер: ['ksc2_vad', 'kkturbo_vad', 'kkturbo_chunk', 'ksc2_chunk', 'mms_vad']


In [ ]:
def make_hyps(use):
    return {f"{a}/{b}": [dedup_full(norm_eval(SRC[n][0][(a, b)].raw, NORM_FILLERS, NORM_NUM)) for n in use] for a, b in keys}

def experiment(use, label):
    sw = [SRC[n][1] for n in use]
    hyps = make_hyps(use)
    mlm = LM if LM is not None else (MLMScorer(MLM_MODEL) if USE_MLM else None)
    t = time.time()
    out, taus, _ = run_cv(files, refs, hyps, sw, mlm=mlm, morph=MORPH, lex=(LM.uni if LM is not None else None), folds=FOLDS)
    rows = summarize(files, refs, out)
    df = pd.DataFrame(rows); df.insert(0, "experiment", label)
    print(f"\n=== {label}  ({time.time()-t:.0f} с)  τ(fold)={taus}")
    print(df[["system","WER","CER","better","worse","same"]].round(4).to_string(index=False))
    keep = [f for f in files if f not in EXCLUDE]
    d2 = pd.DataFrame(summarize(keep, refs, out)); print(f"-- {len(EXCLUDE)} файлсыз ({len(keep)} файл):"); print(d2[d2.system.isin(['raw','vote','crf_auto','oracle'])][["system","WER","CER","better","worse","same"]].round(4).to_string(index=False))
    return df, out, taus

# CSE морфология: qaz_stems_unik_edit.xlsx + qaz_ending_5994.xlsx (Drive ішінен автоматты іздейді)
USE_CSE = True
MORPH = default_morph
if USE_CSE:
    _st = glob.glob(os.path.join(BASE, "**", "qaz_stems_unik_edit.xlsx"), recursive=True) + glob.glob("/content/drive/MyDrive/**/qaz_stems_unik_edit.xlsx", recursive=True)
    _en = glob.glob(os.path.join(BASE, "**", "qaz_ending_5994.xlsx"), recursive=True) + glob.glob("/content/drive/MyDrive/**/qaz_ending_5994.xlsx", recursive=True)
    if _st and _en:
        from collections import Counter
        _cnt = Counter(w for f in files for h in make_hyps(names) [f] for w in h.split())   # тек гипотезалардан (ref қолданылмайды)
        TAGGER = Tagger.build(_st[0], _en[0], _cnt)
        MORPH = make_cse_morph(TAGGER, default_morph)
        print("CSE морфология қосылды:", _st[0])
    else:
        print("CSE файлдары табылмады → жай морфология. Күтілген аттар: qaz_stems_unik_edit.xlsx, qaz_ending_5994.xlsx")

# Биграмма ЯМ: қазақша мәтін корпусы (Drive ішінен іздейді). Тест ref-тері корпусқа кірмегеніне көз жеткізіңіз.
USE_LM = True
LM_SOURCE = "ksc2"     # "ksc2" (ауызша, ksc2_lm_corpus.txt) | "news" (corpus_kaz_285000*.txt) | "both"
LM = None
def _find(pat):
    return glob.glob(os.path.join(BASE, "**", pat), recursive=True) + glob.glob("/content/drive/MyDrive/**/" + pat, recursive=True)
if USE_LM:
    _src = []
    if LM_SOURCE in ("ksc2", "both"): _src += _find("ksc2_lm_corpus*.txt")[:1]
    if LM_SOURCE in ("news", "both"): _src += _find("corpus_kaz_285000*.txt")[:1]
    if _src:
        # ЖОЮ: тест ref-терімен 6-грамм қиылысатын жолдарды корпустан алып тастаймыз (data leakage болмасын)
        _norm = lambda t: norm_eval(t, NORM_FILLERS, NORM_NUM)
        def _grams(w, n=6): return {" ".join(w[i:i+n]) for i in range(len(w) - n + 1)}
        _bad = set()
        for f in files: _bad |= _grams(refs[f].split())
        _tmp = "/content/lm_corpus_filtered.txt"; kept = dropped = 0
        with open(_tmp, "w", encoding="utf-8") as fo:
            for p in _src:
                for line in open(p, encoding="utf-8", errors="ignore"):
                    if _grams(_norm(line).split()) & _bad: dropped += 1; continue
                    fo.write(line if line.endswith("\n") else line + "\n"); kept += 1
        print(f"ЯМ корпусы: {_src} | қалды {kept}, тест ref-пен қиылысқаны алынды: {dropped}")
        _vocab = {w for f in files for h in make_hyps(names)[f] for w in h.split()}
        LM = BigramLM(_tmp, _vocab, _norm)
        print("ЯМ қосылды, көзі:", LM_SOURCE)
    else:
        print("ЯМ корпусы табылмады (ksc2_lm_corpus.txt / corpus_kaz_285000*.txt) → ЯМ өшірулі")

# Әр жүйенің жеке WER/CER (бірдей нормализация, жаңа ref)
_rows = []
for n in names:
    h = {f: dedup_full(norm_eval(SRC[n][0][tuple(f.split("/"))].raw, NORM_FILLERS, NORM_NUM)) for f in files}
    N = sum(len(refs[f].split()) for f in files); e = sum(errs(refs[f], h[f]) for f in files)
    keep = [f for f in files if f not in EXCLUDE]
    _rows.append(dict(system=n, WER=e / N, CER=jiwer.cer([refs[f] for f in files], [h[f] or "x" for f in files])))
SINGLE = pd.DataFrame(_rows).round(4); print("\nЖеке жүйелер (dedup + бірдей нормализация):"); print(SINGLE.to_string(index=False))

ALL, SAVED = [], {}
for label, use in [("ALL", names)] + ([("without_ctc2", [n for n in names if n != "ctc2_vad"])] if "ctc2_vad" in names else []) + ([("first3", names[:3])] if len(names) > 2 else []):
    df, out, taus = experiment(use, label)
    ALL.append(df); SAVED[label] = (out, taus)
res = pd.concat(ALL)

CSE файлдары табылмады → жай морфология. Күтілген аттар: qaz_stems_unik_edit.xlsx, qaz_ending_5994.xlsx
ЯМ корпусы: ['/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/ksc2_lm_corpus.txt'] | қалды 117789, тест ref-пен қиылысқаны алынды: 27
ЯМ: 117789 сөйлем, 1315351 сөз, сөздік 142852, биграмма 655675
ЯМ қосылды, көзі: ksc2

Жеке жүйелер (dedup + бірдей нормализация):
       system    WER    CER
     ksc2_vad 0.2019 0.1054
  kkturbo_vad 0.2114 0.1178
kkturbo_chunk 0.2852 0.1921
   ksc2_chunk 0.4566 0.3752
      mms_vad 0.4983 0.1701
  features 20/100
  features 40/100
  features 60/100
  features 80/100
  features 100/100
  fold 1/5 τ=0.4
  fold 2/5 τ=0.4
  fold 3/5 τ=0.4
  fold 4/5 τ=0.4
  fold 5/5 τ=0.5

=== ALL  (156 с)  τ(fold)=[0.4, 0.4, 0.4, 0.4, 0.5]
  system    WER    CER  better  worse  same
     raw 0.2019 0.1054       0      0   100
    vote 0.1828 0.0903      73     17    10
crf_auto 0.1778 0.0873      76     12    12
 crf_0.4 0.1777 0.0873      76     12    12
 crf_0.5

In [ ]:
# Ең жақсы нұсқа: файл деңгейінде (RAW=негізгі жүйе, dedup-тен кейін)
lab = "ALL"
out = SAVED[lab][0]
pf = pd.DataFrame({"file": files, "ref": [refs[f] for f in files],
                   "raw": [out["raw"][f] for f in files], "vote": [out["vote"][f] for f in files],
                   "mapp2_crf": [out["crf_auto"][f] for f in files], "oracle": [out["oracle"][f] for f in files]})
for c in ["raw", "vote", "mapp2_crf", "oracle"]:
    pf["err_" + c] = [errs(r, h) for r, h in zip(pf.ref, pf[c])]
pf["n_words"] = pf.ref.str.split().str.len()
print("Файл деңгейі (MAPP-2 vs RAW): жақсарды", int((pf.err_mapp2_crf < pf.err_raw).sum()),
      "| нашарлады", int((pf.err_mapp2_crf > pf.err_raw).sum()),
      "| өзгермеді", int((pf.err_mapp2_crf == pf.err_raw).sum()))
OUT_XLSX = os.path.join(OUT_DIR, "mapp2_comparison.xlsx")
with pd.ExcelWriter(OUT_XLSX) as w:
    res.round(4).to_excel(w, sheet_name="summary", index=False)
    SINGLE.to_excel(w, sheet_name="single_systems", index=False)
    pf.to_excel(w, sheet_name="per_file", index=False)
    pd.DataFrame({"fold": range(1, FOLDS+1), "tau": SAVED["ALL"][1]}).to_excel(w, sheet_name="tau", index=False)
print("Сақталды:", OUT_XLSX)

Файл деңгейі (MAPP-2 vs RAW): жақсарды 76 | нашарлады 12 | өзгермеді 12
Сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/mapp2/mapp2_comparison.xlsx
